Created the query on the mock catalog based on the following:
```
SELECT 
    floor(source_id / ((power(2, 35)) * (power(4, (12 - 3))))) AS healpix_,
    floor((phot_g_mean_mag_ext - 3) / 0.25) AS phot_g_mean_mag_,
    floor(((phot_g_mean_mag_ext - phot_rp_mean_mag_ext) - (-0.4)) / 0.1) AS g_rp_,
    COUNT(*) AS n
FROM (
    SELECT GAVO_NORMAL_RANDOM(phot_g_mean_mag, phot_g_mean_mag_error) + a0*A0_1_gaia_g as phot_g_mean_mag_ext,
           GAVO_NORMAL_RANDOM(phot_rp_mean_mag, phot_rp_mean_mag_error) + a0*A0_1_gaia_rp as phot_rp_mean_mag_ext,
           source_id
    FROM gedr3mock.main
    JOIN gedr3mock.parsec_props
    USING (index_parsec)) as mock
WHERE phot_g_mean_mag_ext - phot_rp_mean_mag_ext > -0.4
      AND phot_g_mean_mag_ext - phot_rp_mean_mag_ext < 2.2
      AND phot_g_mean_mag_ext > 3 
      AND phot_g_mean_mag_ext < 20
GROUP BY healpix_, phot_g_mean_mag_, g_rp_
```
```
SELECT 
    floor(source_id / ((power(2, 35)) * (power(4, (12 - 3))))) AS healpix_,
    floor((phot_g_mean_mag_ext - 3) / 0.25) AS phot_g_mean_mag_,
    floor(((phot_g_mean_mag_ext - phot_rp_mean_mag_ext) - (-0.4)) / 0.1) AS g_rp_,
    COUNT(*) AS k
FROM (
    SELECT GAVO_NORMAL_RANDOM(phot_g_mean_mag, phot_g_mean_mag_error) + a0*A0_1_gaia_g as phot_g_mean_mag_ext,
           GAVO_NORMAL_RANDOM(phot_rp_mean_mag, phot_rp_mean_mag_error) + a0*A0_1_gaia_rp as phot_rp_mean_mag_ext,
           source_id,
           parallax
    FROM gedr3mock.main
    JOIN gedr3mock.parsec_props
    USING (index_parsec)) as mock
WHERE phot_g_mean_mag_ext - phot_rp_mean_mag_ext > -0.4
      AND phot_g_mean_mag_ext - phot_rp_mean_mag_ext < 2.2
      AND phot_g_mean_mag_ext > 3 
      AND phot_g_mean_mag_ext < 20 AND parallax > 10
GROUP BY healpix_, phot_g_mean_mag_, g_rp_
```


In [1]:
from astropy.table import join, Table

mock_select = join(Table.read('../../gaiaedr3_mock_counts_order3_smallbins.fits', hdu=1),
                   Table.read('../../gaiaedr3_mock_counts_order3_smallbins.fits', hdu=2),
                   join_type='left', keys=['healpix_', 'phot_g_mean_mag_', 'g_rp_'])

mock_select = mock_select.filled(0)

mock_select['healpix_'] = mock_select['healpix_'].astype(int)
mock_select['phot_g_mean_mag_'] = mock_select['phot_g_mean_mag_'].astype(int)
mock_select['g_rp_'] = mock_select['g_rp_'].astype(int)

mock_select['n'][mock_select['k'] > mock_select['n']] = mock_select['k'][mock_select['k'] > mock_select['n']]

mock_select['k'].name = 'km'
mock_select['n'].name = 'nm'

Used my subselection code on sdssdb to create the counts on the SDSS/Gaia data.
```
cols = ['healpix', catalogdb.Gaia_DR3.phot_g_mean_mag, catalogdb.Gaia_DR3.g_rp]
binning = [ 3,                # order of healpix such that nside = 2^order
           [3, 20, 0.25],     # bins in phot_g_mean_mag, indexed by [min, max, delta]
           [-0.4, 2.2, 0.1]]  # bins in g_rp, indexed by [min, max, delta]

query = obs_query(carton='mwm_snc_100pc', all_subcartons=True)

subqeury = create_subquery(query, cols, binning)

subselection_query = create_subselection_query(subqeury, cols)

df = subselection_query_to_df(subselection_query, 'test_100pc_SF_no_plx_order3_smallbins', cols, binning)
```

In [2]:
subSF = Table.read('/Users/imedan/.gaiaunlimited/test_100pc_SF_no_plx_order3_smallbins.csv', header_start=1)

Join the tables. Don't include the k in subSF. Eventually want this to be DR19?

In [3]:
subSF_mock = join(subSF[['healpix_', 'phot_g_mean_mag_', 'g_rp_', 'n']],
                  mock_select,
                  join_type='left',
                  keys=['healpix_', 'phot_g_mean_mag_', 'g_rp_'])

subSF_mock = subSF_mock.filled(0)

In [4]:
bin_str = "{'healpix': 3, 'phot_g_mean_mag': [3, 20, 0.25], 'g_rp': [-0.4, 2.2, 0.1]}"

with open('../src/snc_sf/sf_files/100pc_SF.csv', "w") as f:
    f.write(f"#{bin_str}\n")
    subSF_mock.write(f, format='csv')
